In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import cv2


# Explore GTSDB structure

In [ ]:
import os

GTSDB_PATH = r"E:\Cardiff Uni\Semester 2\CMT307 Applied Machine Learning\Assessment Deep learning\Detection\archive\GTSDB_Train_and_Test"

# Check Train folder contents
train_path = os.path.join(GTSDB_PATH, "Train")
all_files = os.listdir(train_path)

# Group by extension
from collections import Counter
extensions = Counter(os.path.splitext(f)[1].lower() for f in all_files)
print("File types in Train/:")
for ext, count in extensions.most_common():
    print(f"  {ext}: {count}")

# Check if there are subdirectories
subdirs = [f for f in all_files if os.path.isdir(os.path.join(train_path, f))]
print(f"\nSubdirectories: {subdirs}")

# If there are subdirs, check inside the first one
if subdirs:
    first_sub = os.path.join(train_path, subdirs[0])
    sub_files = os.listdir(first_sub)
    sub_ext = Counter(os.path.splitext(f)[1].lower() for f in sub_files)
    print(f"\nInside {subdirs[0]}/:")
    for ext, count in sub_ext.most_common():
        print(f"  {ext}: {count}")

# Show first 10 files
print(f"\nFirst 10 files:")
for f in sorted(all_files)[:10]:
    print(f"  {f}")

In [ ]:
GTSDB_PATH = r"E:\Cardiff Uni\Semester 2\CMT307 Applied Machine Learning\Assessment Deep learning\Detection\archive\GTSDB_Train_and_Test"

train_img_path = os.path.join(GTSDB_PATH, "Train", "images")
train_lbl_path = os.path.join(GTSDB_PATH, "Train", "labels")
test_img_path = os.path.join(GTSDB_PATH, "Test", "images")
test_lbl_path = os.path.join(GTSDB_PATH, "Test", "labels")

# Count files
train_imgs = sorted([f for f in os.listdir(train_img_path) if f.endswith(('.jpg', '.png'))])
train_lbls = sorted([f for f in os.listdir(train_lbl_path) if f.endswith('.txt')])
test_imgs = sorted([f for f in os.listdir(test_img_path) if f.endswith(('.jpg', '.png'))])
test_lbls = sorted([f for f in os.listdir(test_lbl_path) if f.endswith('.txt')])

print(f"Train — Images: {len(train_imgs)}, Labels: {len(train_lbls)}")
print(f"Test  — Images: {len(test_imgs)}, Labels: {len(test_lbls)}")

# Show sample label
print(f"\nSample label ({train_lbls[0]}):")
with open(os.path.join(train_lbl_path, train_lbls[0]), 'r') as f:
    print(f.read())

# Check image size
img = cv2.imread(os.path.join(train_img_path, train_imgs[0]))
print(f"Sample image size: {img.shape}")

# Show a few more labels to understand the format
print("\nFirst 5 label files:")
for lbl in train_lbls[:5]:
    with open(os.path.join(train_lbl_path, lbl), 'r') as f:
        content = f.read().strip()
    print(f"  {lbl}: {content[:100]}")

# Visualize some GTSDB images with bounding boxes

In [ ]:
train_img_path = os.path.join(GTSDB_PATH, "Train", "images")
train_lbl_path = os.path.join(GTSDB_PATH, "Train", "labels")

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

# Pick images that have labels
labeled_images = sorted([f for f in os.listdir(train_lbl_path) if f.endswith('.txt')])

for idx, lbl_file in enumerate(labeled_images[:6]):
    img_name = lbl_file.replace('.txt', '.jpg')
    img = cv2.imread(os.path.join(train_img_path, img_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    axes[idx].imshow(img)
    
    # Draw bounding boxes
    with open(os.path.join(train_lbl_path, lbl_file), 'r') as f:
        for line in f.readlines():
            parts = line.strip().split()
            cls_id = int(parts[0])
            x_center, y_center, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            
            # Convert from YOLO format to pixel coordinates
            x1 = int((x_center - bw/2) * w)
            y1 = int((y_center - bh/2) * h)
            box_w = int(bw * w)
            box_h = int(bh * h)
            
            rect = patches.Rectangle((x1, y1), box_w, box_h, 
                                      linewidth=2, edgecolor='lime', facecolor='none')
            axes[idx].add_patch(rect)
            axes[idx].text(x1, y1-5, f'Class {cls_id}', color='lime', fontsize=8,
                          fontweight='bold', bbox=dict(boxstyle='round,pad=0.2', 
                          facecolor='black', alpha=0.7))
    
    axes[idx].set_title(img_name, fontsize=10)
    axes[idx].axis('off')

plt.suptitle('GTSDB — Training Images with Bounding Boxes', fontsize=14)
plt.tight_layout()
plt.savefig('gtsdb_samples.png', dpi=150, bbox_inches='tight')
plt.show()

# Check class distribution in detection dataset

In [ ]:
from collections import Counter

all_classes = []
for lbl_file in os.listdir(train_lbl_path):
    if lbl_file.endswith('.txt'):
        with open(os.path.join(train_lbl_path, lbl_file), 'r') as f:
            for line in f.readlines():
                cls_id = int(line.strip().split()[0])
                all_classes.append(cls_id)

class_counts = Counter(all_classes)
print(f"Total bounding boxes in training: {len(all_classes)}")
print(f"Unique classes: {len(class_counts)}")
print(f"\nClass distribution:")
for cls_id in sorted(class_counts.keys()):
    print(f"  Class {cls_id:>2}: {class_counts[cls_id]:>4} boxes")

# Create YOLO config and train

In [ ]:
# Create the YAML config file for YOLOv8
yaml_content = f"""
train: {os.path.join(GTSDB_PATH, 'Train', 'images').replace(os.sep, '/')}
val: {os.path.join(GTSDB_PATH, 'Test', 'images').replace(os.sep, '/')}

nc: 43

names:
  0: speed_limit_20
  1: speed_limit_30
  2: speed_limit_50
  3: speed_limit_60
  4: speed_limit_70
  5: speed_limit_80
  6: end_speed_limit_80
  7: speed_limit_100
  8: speed_limit_120
  9: no_overtaking
  10: no_overtaking_trucks
  11: priority_next_intersection
  12: priority_road
  13: yield
  14: stop
  15: no_vehicles
  16: no_trucks
  17: no_entry
  18: general_caution
  19: dangerous_curve_left
  20: dangerous_curve_right
  21: double_curve
  22: bumpy_road
  23: slippery_road
  24: road_narrows_right
  25: road_work
  26: traffic_signals
  27: pedestrians
  28: children_crossing
  29: bicycles_crossing
  30: beware_ice_snow
  31: wild_animals_crossing
  32: end_all_limits
  33: right_turn_ahead
  34: left_turn_ahead
  35: ahead_only
  36: go_straight_or_right
  37: go_straight_or_left
  38: keep_right
  39: keep_left
  40: roundabout
  41: end_no_overtaking
  42: end_no_overtaking_trucks
"""

config_path = os.path.join(GTSDB_PATH, 'gtsdb.yaml')
with open(config_path, 'w') as f:
    f.write(yaml_content)

print(f"Config saved to: {config_path}")

# Train YOLOv8

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 nano (smallest, fastest to train)
model_yolo = YOLO('yolov8n.pt')

# Train
results = model_yolo.train(
    data=config_path,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    project='gtsdb_detection',
    name='yolov8n_run1',
    device=0,          # Use your RTX 4060
    workers=0,         # Windows compatibility
    verbose=True
)

# Evaluate on test set

In [ ]:
# Evaluate on test set
metrics = model_yolo.val(
    data=config_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0
)

print(f"\n{'='*60}")
print(f"YOLOv8 DETECTION — TEST RESULTS")
print(f"{'='*60}")
print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")

# Visualize detections on test images

In [ ]:
test_img_path = os.path.join(GTSDB_PATH, "Test", "images")
test_images = sorted([f for f in os.listdir(test_img_path) if f.endswith('.jpg')])

fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

for idx, img_name in enumerate(test_images[10:16]):
    img_path = os.path.join(test_img_path, img_name)
    
    # Run inference
    results = model_yolo.predict(img_path, conf=0.25, imgsz=640, device=0, verbose=False)
    
    # Plot result
    result_img = results[0].plot()
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(result_img)
    axes[idx].set_title(f'{img_name} — {len(results[0].boxes)} detections', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('YOLOv8 Detections on Test Images', fontsize=14)
plt.tight_layout()
plt.savefig('yolov8_detections.png', dpi=150, bbox_inches='tight')
plt.show()

# Convert labels to single-class detection

In [ ]:
import shutil


# Create new label directories
for split in ['Train', 'Test']:
    src_labels = os.path.join(GTSDB_PATH, split, 'labels')
    new_labels = os.path.join(GTSDB_PATH, split, 'labels_single')
    os.makedirs(new_labels, exist_ok=True)
    
    count = 0
    for lbl_file in os.listdir(src_labels):
        if lbl_file.endswith('.txt'):
            with open(os.path.join(src_labels, lbl_file), 'r') as f:
                lines = f.readlines()
            
            # Replace all class IDs with 0 (single class: "traffic_sign")
            new_lines = []
            for line in lines:
                parts = line.strip().split()
                if len(parts) == 5:
                    parts[0] = '0'  # all signs become class 0
                    new_lines.append(' '.join(parts) + '\n')
            
            with open(os.path.join(new_labels, lbl_file), 'w') as f:
                f.writelines(new_lines)
            count += 1
    
    print(f"{split}: converted {count} label files to single-class")

# Create new YAML config for single-class

In [ ]:
# We need to point to the new labels
# YOLOv8 expects labels in a folder called 'labels' next to 'images'
# So we'll rename folders

for split in ['Train', 'Test']:
    original_labels = os.path.join(GTSDB_PATH, split, 'labels')
    backup_labels = os.path.join(GTSDB_PATH, split, 'labels_43class')
    single_labels = os.path.join(GTSDB_PATH, split, 'labels_single')
    
    # Backup original
    if not os.path.exists(backup_labels):
        os.rename(original_labels, backup_labels)
        print(f"{split}: backed up original labels to labels_43class/")
    
    # Make single-class the active labels folder
    if not os.path.exists(original_labels):
        os.rename(single_labels, original_labels)
        print(f"{split}: activated single-class labels")

# Create new config
yaml_single = f"""
train: {os.path.join(GTSDB_PATH, 'Train', 'images').replace(os.sep, '/')}
val: {os.path.join(GTSDB_PATH, 'Test', 'images').replace(os.sep, '/')}

nc: 1

names:
  0: traffic_sign
"""

config_single_path = os.path.join(GTSDB_PATH, 'gtsdb_single.yaml')
with open(config_single_path, 'w') as f:
    f.write(yaml_single)

print(f"\nSingle-class config saved to: {config_single_path}")

# Train YOLOv8 single-class

In [ ]:
from ultralytics import YOLO

model_yolo_single = YOLO('yolov8n.pt')

results_single = model_yolo_single.train(
    data=config_single_path,
    epochs=80,
    imgsz=640,
    batch=16,
    patience=15,
    save=True,
    project='gtsdb_detection',
    name='yolov8n_single_class',
    device=0,
    workers=0,
    augment=True,
    verbose=True
)

#  Evaluate single-class detection

In [ ]:
metrics_single = model_yolo_single.val(
    data=config_single_path,
    split='val',
    imgsz=640,
    batch=16,
    device=0,
    workers=0
)

print(f"\n{'='*60}")
print(f"YOLOv8 SINGLE-CLASS DETECTION — TEST RESULTS")
print(f"{'='*60}")
print(f"mAP@50:      {metrics_single.box.map50:.4f}")
print(f"mAP@50-95:   {metrics_single.box.map:.4f}")
print(f"Precision:   {metrics_single.box.mp:.4f}")
print(f"Recall:      {metrics_single.box.mr:.4f}")

#  Visualize single-class detections

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 10))
axes = axes.flatten()

test_img_path = os.path.join(GTSDB_PATH, "Test", "images")
test_images = sorted([f for f in os.listdir(test_img_path) if f.endswith('.jpg')])

# Pick images that are likely to have signs
for idx, img_name in enumerate(test_images[0:6]):
    img_path = os.path.join(test_img_path, img_name)
    
    results = model_yolo_single.predict(img_path, conf=0.25, imgsz=640, device=0, verbose=False)
    
    result_img = results[0].plot()
    result_img = cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB)
    
    axes[idx].imshow(result_img)
    axes[idx].set_title(f'{img_name} — {len(results[0].boxes)} detections', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('YOLOv8 Single-Class Detection on Test Images', fontsize=14)
plt.tight_layout()
plt.savefig('yolov8_single_class_detections.png', dpi=150, bbox_inches='tight')
plt.show()

# Full Detection + Recognition Pipeline

In [ ]:
import torch
import torch.nn as nn
import cv2
import numpy as np
from torchvision import transforms
from ultralytics import YOLO

# Load both models
# 1. YOLO detector
detector = YOLO(r"E:\Cardiff Uni\Semester 2\CMT307 Applied Machine Learning\Assessment Deep learning\runs\detect\gtsdb_detection\yolov8n_single_class\weights\best.pt")

# 2. Baseline CNN classifier (reload architecture + weights)
class BaselineCNN(nn.Module):
    def __init__(self, num_classes=43):
        super(BaselineCNN, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.BatchNorm2d(32),
            nn.ReLU(inplace=True), nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.BatchNorm2d(64),
            nn.ReLU(inplace=True), nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128),
            nn.ReLU(inplace=True), nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2, 2), nn.Dropout2d(0.25))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128 * 6 * 6, 512),
            nn.ReLU(inplace=True), nn.Dropout(0.5), nn.Linear(512, num_classes))
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.classifier(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
classifier = BaselineCNN(num_classes=43).to(device)
classifier.load_state_dict(torch.load('best_baseline_cnn.pth', map_location=device))
classifier.eval()

# Class names for display
CLASS_NAMES = [
    'Speed limit 20', 'Speed limit 30', 'Speed limit 50', 'Speed limit 60',
    'Speed limit 70', 'Speed limit 80', 'End speed limit 80', 'Speed limit 100',
    'Speed limit 120', 'No overtaking', 'No overtaking trucks',
    'Priority at next intersection', 'Priority road', 'Yield', 'Stop',
    'No vehicles', 'No trucks', 'No entry', 'General caution',
    'Dangerous curve left', 'Dangerous curve right', 'Double curve',
    'Bumpy road', 'Slippery road', 'Road narrows right', 'Road work',
    'Traffic signals', 'Pedestrians', 'Children crossing', 'Bicycles crossing',
    'Beware ice/snow', 'Wild animals crossing', 'End all limits',
    'Right turn ahead', 'Left turn ahead', 'Ahead only',
    'Go straight or right', 'Go straight or left', 'Keep right',
    'Keep left', 'Roundabout', 'End no overtaking', 'End no overtaking trucks'
]

# Preprocessing for classifier (same as training)
classify_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((48, 48)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


def detect_and_classify(image_path, detector, classifier, conf_threshold=0.25):
    """Full pipeline: detect signs then classify each one."""
    
    # Step 1: Detect signs with YOLO
    detections = detector.predict(image_path, conf=conf_threshold, imgsz=640, device=0, verbose=False)
    
    # Read original image
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    results = []
    
    if len(detections[0].boxes) == 0:
        return img_rgb, results
    
    # Step 2: For each detected sign, crop and classify
    for box in detections[0].boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy().astype(int)
        det_conf = box.conf[0].cpu().numpy()
        
        # Add padding around the crop (10%)
        h, w = img.shape[:2]
        pad_x = int((x2 - x1) * 0.1)
        pad_y = int((y2 - y1) * 0.1)
        x1_pad = max(0, x1 - pad_x)
        y1_pad = max(0, y1 - pad_y)
        x2_pad = min(w, x2 + pad_x)
        y2_pad = min(h, y2 + pad_y)
        
        # Crop the sign
        crop = img_rgb[y1_pad:y2_pad, x1_pad:x2_pad]
        
        if crop.size == 0:
            continue
        
        # Apply CLAHE (same as training)
        crop_lab = cv2.cvtColor(crop, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        crop_lab[:, :, 0] = clahe.apply(crop_lab[:, :, 0])
        crop_enhanced = cv2.cvtColor(crop_lab, cv2.COLOR_LAB2RGB)
        
        # Classify
        input_tensor = classify_transform(crop_enhanced).unsqueeze(0).to(device)
        with torch.no_grad():
            output = classifier(input_tensor)
            probabilities = torch.softmax(output, dim=1)
            cls_conf, cls_pred = probabilities.max(1)
        
        class_id = cls_pred.item()
        class_conf = cls_conf.item()
        class_name = CLASS_NAMES[class_id]
        
        results.append({
            'bbox': (x1, y1, x2, y2),
            'det_conf': float(det_conf),
            'class_id': class_id,
            'class_name': class_name,
            'class_conf': float(class_conf)
        })
    
    return img_rgb, results


def visualize_pipeline(image_path, detector, classifier, conf_threshold=0.25):
    """Visualize full pipeline results."""
    img_rgb, results = detect_and_classify(image_path, detector, classifier, conf_threshold)
    
    fig, ax = plt.subplots(1, 1, figsize=(16, 9))
    ax.imshow(img_rgb)
    
    for r in results:
        x1, y1, x2, y2 = r['bbox']
        
        # Draw bounding box
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        ax.add_patch(rect)
        
        # Label with class name and confidence
        label = f"{r['class_name']} ({r['class_conf']:.0%})"
        ax.text(x1, y1 - 8, label, color='white', fontsize=9,
                fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='green', alpha=0.8))
    
    ax.set_title(f'Pipeline Result — {len(results)} sign(s) detected', fontsize=14)
    ax.axis('off')
    plt.tight_layout()
    return fig

# Run pipeline on test images

In [ ]:
test_img_path = os.path.join(GTSDB_PATH, "Test", "images")
test_images = sorted([f for f in os.listdir(test_img_path) if f.endswith('.jpg')])

# Run on 8 diverse images
fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

# Pick images likely to have signs
sample_indices = [1, 2, 3, 4, 5, 10, 15, 20]

for idx, img_idx in enumerate(sample_indices):
    img_path = os.path.join(test_img_path, test_images[img_idx])
    img_rgb, results = detect_and_classify(img_path, detector, classifier)
    
    axes[idx].imshow(img_rgb)
    
    for r in results:
        x1, y1, x2, y2 = r['bbox']
        rect = patches.Rectangle((x1, y1), x2-x1, y2-y1,
                                  linewidth=2, edgecolor='lime', facecolor='none')
        axes[idx].add_patch(rect)
        label = f"{r['class_name']}\n({r['class_conf']:.0%})"
        axes[idx].text(x1, y1 - 8, label, color='white', fontsize=7,
                      fontweight='bold',
                      bbox=dict(boxstyle='round,pad=0.2', facecolor='green', alpha=0.8))
    
    axes[idx].set_title(f'{test_images[img_idx]} — {len(results)} sign(s)', fontsize=9)
    axes[idx].axis('off')

plt.suptitle('Full Pipeline: YOLOv8 Detection → Baseline CNN Classification', fontsize=14)
plt.tight_layout()
plt.savefig('full_pipeline_results.png', dpi=150, bbox_inches='tight')
plt.show()

# Pipeline summary

In [ ]:
print("=" * 70)
print("COMPLETE PROJECT SUMMARY")
print("=" * 70)

print("\n PHASE 1: EXPLORATORY DATA ANALYSIS")
print(f"   GTSRB: 39,209 train / 12,630 test images, 43 classes")
print(f"   GTSDB: 600 train / 300 test full scene images")
print(f"   Key finding: 10.7x class imbalance, tiny images (mean 50×50px)")

print("\n PHASE 2: CLASSIFICATION (RECOGNITION)")
print(f"   {'Model':<25} {'Test Accuracy':>15}")
print(f"   {'-'*40}")
print(f"   {'Baseline CNN':<25} {'98.23%':>15}  ← Best")
print(f"   {'ResNet-18 v2':<25} {'97.51%':>15}")
print(f"   {'MobileNetV2 v2':<25} {'95.96%':>15}")

print("\n PHASE 3: DETECTION")
print(f"   {'Model':<25} {'mAP@50':>10} {'Precision':>12} {'Recall':>10}")
print(f"   {'-'*57}")
print(f"   {'YOLOv8 (43 classes)':<25} {'5.77%':>10} {'20.27%':>12} {'11.24%':>10}")
print(f"   {'YOLOv8 (single class)':<25} {'91.42%':>10} {'87.57%':>12} {'83.66%':>10}")

print("\n PHASE 4: END-TO-END PIPELINE")
print(f"   YOLOv8 detects signs → Baseline CNN classifies (43 classes)")
print(f"   Two-stage architecture combining best detection + best classification")

print("\n" + "=" * 70)